In [1]:
# ============================================================
# 0_FNS
# ============================================================

# ----------------------------######----------------------------#
#   _aiff_0901_cover2mp3_GET_safe_folder                         #
# ----------------------------######----------------------------#

import os
from tqdm import tqdm
from pydub import AudioSegment

from mutagen.aiff import AIFF
from mutagen.mp3 import MP3
from mutagen.id3 import (
    ID3, ID3NoHeaderError,
    APIC, TIT2, TPE1, TALB, TCON, COMM, TDRC, TKEY, TBPM
)


AIFF_EXTS = (".aiff", ".aif")


def _aiff_0901_cover2mp3_GET_safe_folder(folder_path,
                                        bitrate="320k",
                                        overwrite_mp3=False,
                                        ask_delete_aiff=True):
    """
    Recursively:
      1) Find all AIFF/AIF in folder_path
      2) Extract embedded cover art (APIC) + ID3 text frames from AIFF
      3) Convert audio to MP3 (DJ-safe: 320k CBR by default)
      4) Wipe any existing MP3 ID3 (prevents 'an ID3 tag already exists')
      5) Write fresh tags + embed the SAME cover art bytes into MP3
      6) Ask (at end) whether to delete original AIFFs

    Output MP3 is created next to original AIFF with same basename.
    """

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"❌ Folder not found: {folder_path}")

    # ---------- collect ----------
    aiff_files = []
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(AIFF_EXTS) and not f.startswith("._") and f != ".DS_Store":
                aiff_files.append(os.path.join(root, f))

    if not aiff_files:
        print("⚠️ No AIFF/AIF files found.")
        return {
            "found": 0, "converted": 0, "skipped_exists": 0, "failed": 0,
            "converted_paths": [], "failed_paths": []
        }

    converted_paths = []
    failed_paths = []
    skipped_exists = 0

    # ---------- helpers ----------
    def _aiff_extract_id3_and_artwork(aiff_path):
        """
        Returns:
          aiff_id3 (mutagen ID3-like object or None),
          artwork_bytes (bytes or None),
          artwork_mime (str or None)
        """
        a = AIFF(aiff_path)

        aiff_id3 = a.tags  # often an ID3 container when AIFF has ID3 chunk
        artwork_bytes = None
        artwork_mime = None

        if aiff_id3:
            try:
                apics = aiff_id3.getall("APIC")
                if apics:
                    artwork_bytes = apics[0].data
                    artwork_mime = apics[0].mime
            except Exception:
                pass

        # fallback: try to find any object with .data that looks like an image
        if (not artwork_bytes) and aiff_id3:
            try:
                for fr in aiff_id3.values():
                    if hasattr(fr, "data") and isinstance(fr.data, (bytes, bytearray)):
                        b = bytes(fr.data)
                        if b[:3] == b"\xff\xd8\xff":  # JPG
                            artwork_bytes = b
                            artwork_mime = "image/jpeg"
                            break
                        if b[:8] == b"\x89PNG\r\n\x1a\n":  # PNG
                            artwork_bytes = b
                            artwork_mime = "image/png"
                            break
            except Exception:
                pass

        return aiff_id3, artwork_bytes, artwork_mime

    def _get_text(id3_obj, frame_key):
        if not id3_obj:
            return None
        try:
            frames = id3_obj.getall(frame_key)
            if not frames:
                return None
            # Most text frames store a list in .text
            fr = frames[0]
            if hasattr(fr, "text") and fr.text:
                v = str(fr.text[0]).strip()
                return v if v else None
        except Exception:
            return None
        return None

    # ---------- process ----------
    for aiff_path in tqdm(aiff_files, desc="🎧 AIFF → MP3"):
        try:
            mp3_path = os.path.splitext(aiff_path)[0] + ".mp3"

            if os.path.isfile(mp3_path) and not overwrite_mp3:
                skipped_exists += 1
                continue

            # ---- extract tags + cover from AIFF ----
            aiff_id3, artwork, artwork_mime = _aiff_extract_id3_and_artwork(aiff_path)

            # ---- convert audio ----
            audio = AudioSegment.from_file(aiff_path, format="aiff")
            audio.export(mp3_path, format="mp3", bitrate=bitrate)

            # ---- wipe ANY existing mp3 id3 header (prevents 'already exists') ----
            try:
                ID3(mp3_path).delete()
            except ID3NoHeaderError:
                pass
            except Exception:
                # if something weird, still try to proceed with a clean add_tags step below
                pass

            mp3 = MP3(mp3_path, ID3=ID3)
            mp3.add_tags()

            # ---- copy core text frames (from AIFF ID3 if present) ----
            v = _get_text(aiff_id3, "TIT2")
            if v: mp3.tags.add(TIT2(encoding=3, text=v))

            v = _get_text(aiff_id3, "TPE1")
            if v: mp3.tags.add(TPE1(encoding=3, text=v))

            v = _get_text(aiff_id3, "TALB")
            if v: mp3.tags.add(TALB(encoding=3, text=v))

            v = _get_text(aiff_id3, "TCON")
            if v: mp3.tags.add(TCON(encoding=3, text=v))

            v = _get_text(aiff_id3, "TBPM")
            if v: mp3.tags.add(TBPM(encoding=3, text=v))

            v = _get_text(aiff_id3, "TKEY")
            if v: mp3.tags.add(TKEY(encoding=3, text=v))

            v = _get_text(aiff_id3, "TDRC")
            if v: mp3.tags.add(TDRC(encoding=3, text=v))

            # comments (keep first)
            if aiff_id3:
                try:
                    comms = aiff_id3.getall("COMM")
                    if comms:
                        txt = ""
                        if hasattr(comms[0], "text") and comms[0].text:
                            txt = str(comms[0].text[0])
                        if txt.strip():
                            mp3.tags.add(COMM(encoding=3, desc="Comment", text=txt.strip()))
                except Exception:
                    pass

            # ---- embed cover into MP3 ----
            if artwork:
                mime = artwork_mime
                if not mime:
                    if artwork[:3] == b"\xff\xd8\xff":
                        mime = "image/jpeg"
                    elif artwork[:8] == b"\x89PNG\r\n\x1a\n":
                        mime = "image/png"
                    else:
                        mime = "image/jpeg"  # safe default

                mp3.tags.add(
                    APIC(
                        encoding=3,
                        mime=mime,
                        type=3,          # front cover
                        desc="Cover",
                        data=artwork
                    )
                )

            mp3.save()
            converted_paths.append(aiff_path)

        except Exception as e:
            failed_paths.append((aiff_path, str(e)))
            print(f"\n❌ Failed: {aiff_path}\n{e}\n")

    # ---------- summary ----------
    print("\n==================== ✅ SUMMARY ====================")
    print(f"Found AIFF:        {len(aiff_files)}")
    print(f"Converted MP3:     {len(converted_paths)}")
    print(f"Skipped (exists):  {skipped_exists}")
    print(f"Failed:            {len(failed_paths)}")

    # ---------- delete prompt ----------
    if ask_delete_aiff and converted_paths:
        resp = input("\n🗑️ After you check MP3s: delete ORIGINAL AIFF files now? (y/N): ").strip().lower()
        if resp == "y":
            deleted = 0
            delete_failed = 0
            for p in tqdm(converted_paths, desc="🗑️ Deleting AIFF"):
                try:
                    os.remove(p)
                    deleted += 1
                except Exception:
                    delete_failed += 1
            print(f"\n✅ Deleted AIFF: {deleted}")
            if delete_failed:
                print(f"⚠️ Could not delete: {delete_failed}")
        else:
            print("ℹ️ AIFF files kept (nothing deleted).")

    return {
        "found": len(aiff_files),
        "converted": len(converted_paths),
        "skipped_exists": skipped_exists,
        "failed": len(failed_paths),
        "converted_paths": converted_paths,
        "failed_paths": failed_paths,
    }


In [2]:
folder = "/Users/yerik/Music/_1_NEW_SOURCE/mp3"

In [3]:

res = _aiff_0901_cover2mp3_GET_safe_folder(
    folder_path=folder,
    bitrate="320k",
    overwrite_mp3=False,
    ask_delete_aiff=True
)


🎧 AIFF → MP3: 100%|██████████████████████████████████████████████████████| 1140/1140 [52:26<00:00,  2.76s/it]



==================== ✅ SUMMARY ====================
Found AIFF:        1140
Converted MP3:     1140
Skipped (exists):  0
Failed:            0



🗑️ After you check MP3s: delete ORIGINAL AIFF files now? (y/N):  y


🗑️ Deleting AIFF: 100%|█████████████████████████████████████████████████| 1140/1140 [00:00<00:00, 9079.46it/s]


✅ Deleted AIFF: 1140
